# EXERCISE 2

In [14]:
import numpy as np
import pytest

### Task 2.1 Implement the DGEMM with matrices as with lists, arrays, and NumPy.

In [15]:
def dgemm_numpy(A, B, C):
    """
    Perform DGEMM: C = C + A * B
    """
    N = A.shape[0]

    for i in range(N):
        for j in range(N):
            for k in range(N):
                C[i, j] += A[i, k] * B[k, j]

    return C

In [1]:
from array import array
import random

def initialize_arrays(N):
    A = [array('d', [random.random() for _ in range(N)]) for _ in range(N)]
    B = [array('d', [random.random() for _ in range(N)]) for _ in range(N)]
    C = [array('d', [0.0 for _ in range(N)]) for _ in range(N)]
    return A, B, C


def dgemm_arrays(A, B, C, N):
    for i in range(N):
        for j in range(N):
            for k in range(N):
                C[i][j] += A[i][k] * B[k][j]
    return C

In [2]:
import random

def dgemm_lists(A, B, C, N):
    for i in range(N):
        for j in range(N):
            for k in range(N):
                C[i][j] += A[i][k] * B[k][j]
    return C


# Initialize matrices
def initialize_lists(N):
    A = [[random.random() for _ in range(N)] for _ in range(N)]
    B = [[random.random() for _ in range(N)] for _ in range(N)]
    C = [[0.0 for _ in range(N)] for _ in range(N)]
    return A, B, C

### Task 2.2 Using pytest develop a unit test for checking the correctness of your implementations.

In [ ]:
from dgemm import dgemm_lists, dgemm_arrays, dgemm_numpy

In [6]:
import pytest
def generate_test_matrices(N):
    np.random.seed(0)
    A = np.random.rand(N, N)
    B = np.random.rand(N, N)
    C = np.zeros((N, N))
    return A, B, C


def test_dgemm_lists():
    N = 3
    A_np, B_np, C_np = generate_test_matrices(N)

    # Convert to lists
    A = A_np.tolist()
    B = B_np.tolist()
    C = C_np.tolist()

    result = dgemm_lists(A, B, C, N)

    expected = A_np @ B_np

    assert np.allclose(result, expected)


def test_dgemm_arrays():
    N = 3
    A_np, B_np, C_np = generate_test_matrices(N)

    # Convert to arrays
    A = [array('d', row) for row in A_np]
    B = [array('d', row) for row in B_np]
    C = [array('d', row) for row in C_np]

    result = dgemm_arrays(A, B, C, N)

    # Convert result back to numpy for comparison
    result_np = np.array([list(row) for row in result])
    expected = A_np @ B_np

    assert np.allclose(result_np, expected)


def test_dgemm_correctness():
    np.random.seed(0)
    N = 4

    A = np.random.rand(N, N).astype(np.float64)
    B = np.random.rand(N, N).astype(np.float64)
    C = np.random.rand(N, N).astype(np.float64)

    C_test = dgemm_numpy(A, B, C.copy())
    C_expected = C + A @ B

    assert np.allclose(C_test, C_expected)

@pytest.mark.parametrize("N", [1, 2, 5, 10])
def test_dgemm_multiple_sizes(N):
    np.random.seed(1)

    A = np.random.rand(N, N).astype(np.float64)
    B = np.random.rand(N, N).astype(np.float64)
    C = np.random.rand(N, N).astype(np.float64)

    C_test = dgemm_numpy(A, B, C.copy())
    C_expected = C + A @ B

    assert np.allclose(C_test, C_expected)

In [7]:
!python -m pytest -v test_dgemm.py

============================= test session starts ==============================
platform darwin -- Python 3.11.14, pytest-9.0.2, pluggy-1.6.0 -- /Users/mridulachandrika/Documents/Schoolwork/HPC-Intro/.venv/bin/python
cachedir: .pytest_cache
rootdir: /Users/mridulachandrika/Documents/Schoolwork/HPC-Intro/Assignment2/EX2
plugins: anyio-4.12.1
collected 7 items                                                              

test_dgemm.py::test_dgemm_lists PASSED                                   [ 14%]
test_dgemm.py::test_dgemm_arrays PASSED                                  [ 28%]
test_dgemm.py::test_dgemm_correctness PASSED                             [ 42%]
test_dgemm.py::test_dgemm_multiple_sizes[1] PASSED                       [ 57%]
test_dgemm.py::test_dgemm_multiple_sizes[2] PASSED                       [ 71%]
test_dgemm.py::test_dgemm_multiple_sizes[5] PASSED                       [ 85%]
test_dgemm.py::test_dgemm_multiple_sizes[10] PASSED                      [100%]

==============

The first test test_dgemm.py tests the correctness of the implementation by generating martices using random double-precision values and computing the equation: C = C + AB using the implemented functions (lists, arrays, and NumPy versions) and checked whether the result matches with the expected values.

The test_dgemm_multiple_sizes test extends this validation by repeating the correctness check for multiple matrix sizes, ensuring that the implementation works consistently across different problem dimensions.

### Task 2.3 Measure the execution time for each approach varying the matrix size. Report the average and error (std. deviation, min/max, or interval of confidence). Answer the question: how does the computational performance, e.g., the std, vary with increasing the size of the matrices, and why so?

In [8]:
import time
from statistics import mean, stdev

In [ ]:
def benchmark(func, A, B, C, N=None, repeats=5):
    times = []

    for _ in range(repeats):
        C_copy = C.copy() if isinstance(C, np.ndarray) else [row[:] for row in C]

        start = time.perf_counter()

        if N is not None:
            func(A, B, C_copy, N)
        else:
            func(A, B, C_copy)

        end = time.perf_counter()
        times.append(end - start)

    return {
        "mean": mean(times),
        "std": stdev(times),
        "min": min(times),
        "max": max(times),
    }


def run_benchmarks():
    sizes = [50, 100, 200]
    repeats = 5

    for N in sizes:
        print(f"\nMatrix size: {N}x{N}")

        # NumPy matrices
        A_np = np.random.rand(N, N)
        B_np = np.random.rand(N, N)
        C_np = np.zeros((N, N))

        # Lists
        A_list = A_np.tolist()
        B_list = B_np.tolist()
        C_list = [[0.0]*N for _ in range(N)]

        # Arrays
        A_arr = [array('d', row) for row in A_np]
        B_arr = [array('d', row) for row in B_np]
        C_arr = [array('d', [0.0]*N) for _ in range(N)]

        # Run benchmarks
        print("Lists:", benchmark(dgemm_lists, A_list, B_list, C_list, N, repeats))
        print("Arrays:", benchmark(dgemm_arrays, A_arr, B_arr, C_arr, N, repeats))
        print("NumPy:", benchmark(dgemm_numpy, A_np, B_np, C_np, None, repeats))


In [10]:
!python benchmark_dgemm.py


Matrix size: 50x50
Lists: {'mean': 0.008446475001983344, 'std': 0.00032133894205688265, 'min': 0.008175417009624653, 'max': 0.008997750002890825}
Arrays: {'mean': 0.016502458392642437, 'std': 0.00034588250245005063, 'min': 0.015961582990712486, 'max': 0.01688458299031481}
NumPy: {'mean': 1.3116802438162268e-05, 'std': 1.4571209809575707e-05, 'min': 5.875001079402864e-06, 'max': 3.9125006878748536e-05}

Matrix size: 100x100
Lists: {'mean': 0.06516781659447587, 'std': 0.000513790168313916, 'min': 0.06485374999465421, 'max': 0.0660784999927273}
Arrays: {'mean': 0.13056797480094245, 'std': 0.0010564859896443795, 'min': 0.12936237500980496, 'max': 0.13203595799859613}
NumPy: {'mean': 3.298340307082981e-05, 'std': 2.955363664655294e-05, 'min': 1.9042010535486042e-05, 'max': 8.583300223108381e-05}

Matrix size: 200x200
Lists: {'mean': 0.5550289247999899, 'std': 0.015165272283579355, 'min': 0.5428065829910338, 'max': 0.5805132079985924}
Arrays: {'mean': 1.0153177000058349, 'std': 0.0050285685

- The execution time of each DGEMM implementation is measured using repeated runs for every matrix size defined in the benchmark. For each size, the computation is performed five times, and the mean execution time and standard deviation are calculated to evaluate performance variability.
- The execution time increases rapidly as the matrix size grows, which is consistent with the 
𝑂(𝑁3)computational complexity of the DGEMM algorithm. For example, doubling the matrix size results in approximately an eightfold increase in runtime, confirming the expected cubic scaling behavior.
- Although the absolute standard deviation increases for larger matrices, the relative variability remains small. This indicates that performance measurements become more stable as the computational workload increases and the influence of system noise becomes less significant.

### Task 2.4 Using the timing information and the number of operations for the DGEMM, calculate the FLOPS/s. How many operations are carried out in DGEMM with N as the matrices dimension? Hint: Think about the number of iterations completed in the loops and the number of flops per iteration. How do the FLOPS/s you measured compare to the theoretical peak of your processor

In [11]:
# ==============================
# BENCHMARK FUNCTION
# ==============================

def benchmark(func, A, B, C, N=None, repeats=5):
    times = []

    for _ in range(repeats):

        # Copy C so each run starts fresh
        if isinstance(C, np.ndarray):
            C_copy = C.copy()
        else:
            C_copy = [row[:] for row in C]

        start = time.perf_counter()

        if N is not None:
            func(A, B, C_copy, N)
        else:
            func(A, B, C_copy)

        end = time.perf_counter()
        times.append(end - start)

    return {
        "mean": mean(times),
        "std": stdev(times),
        "min": min(times),
        "max": max(times),
    }


# ==============================
# FLOPS CALCULATION
# ==============================

def compute_flops(N, time_seconds):
    total_flops = 2 * (N ** 3)
    flops_per_sec = total_flops / time_seconds
    gflops = flops_per_sec / 1e9
    return flops_per_sec, gflops


# ==============================
# MAIN BENCHMARK
# ==============================

def main():

    sizes = [50, 100, 200]
    repeats = 5

    # Change this to your CPU frequency
    cpu_freq_ghz = 3.0
    theoretical_peak_gflops = cpu_freq_ghz  # assuming 1 flop per cycle

    print(f"Theoretical Peak (per core, 1 flop/cycle): {theoretical_peak_gflops} GFLOPS\n")

    for N in sizes:

        print("=" * 50)
        print(f"Matrix size: {N} x {N}")

        # NumPy matrices
        A_np = np.random.rand(N, N)
        B_np = np.random.rand(N, N)
        C_np = np.zeros((N, N))

        # Lists
        A_list = A_np.tolist()
        B_list = B_np.tolist()
        C_list = [[0.0]*N for _ in range(N)]

        # Arrays
        A_arr = [array('d', row) for row in A_np]
        B_arr = [array('d', row) for row in B_np]
        C_arr = [array('d', [0.0]*N) for _ in range(N)]

        # Run benchmarks
        stats_lists = benchmark(dgemm_lists, A_list, B_list, C_list, N, repeats)
        stats_arrays = benchmark(dgemm_arrays, A_arr, B_arr, C_arr, N, repeats)
        stats_numpy = benchmark(dgemm_numpy, A_np, B_np, C_np, None, repeats)

        for name, stats in [("Lists", stats_lists),
                            ("Arrays", stats_arrays),
                            ("NumPy", stats_numpy)]:

            flops, gflops = compute_flops(N, stats["mean"])

            print(f"\n{name}:")
            print(f"  Mean Time: {stats['mean']:.6f} s")
            print(f"  Std Dev  : {stats['std']:.6f} s")
            print(f"  GFLOPS   : {gflops:.6f}")
            print(f"  % of Peak: {(gflops / theoretical_peak_gflops) * 100:.4f} %")

In [12]:
!python benchmark_dgemm_flops.py 

Theoretical Peak (per core, 1 flop/cycle): 3.0 GFLOPS

Matrix size: 50 x 50

Lists:
  Mean Time: 0.008270 s
  Std Dev  : 0.000191 s
  GFLOPS   : 0.030231
  % of Peak: 1.0077 %

Arrays:
  Mean Time: 0.016307 s
  Std Dev  : 0.000164 s
  GFLOPS   : 0.015331
  % of Peak: 0.5110 %

NumPy:
  Mean Time: 0.000012 s
  Std Dev  : 0.000011 s
  GFLOPS   : 21.613966
  % of Peak: 720.4655 %
Matrix size: 100 x 100

Lists:
  Mean Time: 0.065172 s
  Std Dev  : 0.000475 s
  GFLOPS   : 0.030688
  % of Peak: 1.0229 %

Arrays:
  Mean Time: 0.130683 s
  Std Dev  : 0.001806 s
  GFLOPS   : 0.015304
  % of Peak: 0.5101 %

NumPy:
  Mean Time: 0.000033 s
  Std Dev  : 0.000029 s
  GFLOPS   : 61.412243
  % of Peak: 2047.0748 %
Matrix size: 200 x 200

Lists:
  Mean Time: 0.555685 s
  Std Dev  : 0.012928 s
  GFLOPS   : 0.028793
  % of Peak: 0.9598 %

Arrays:
  Mean Time: 1.004858 s
  Std Dev  : 0.004316 s
  GFLOPS   : 0.015923
  % of Peak: 0.5308 %

NumPy:
  Mean Time: 0.000103 s
  Std Dev  : 0.000040 s
  GFLOPS   :

- The pure Python implementations (lists and arrays) achieve only about 1% of the theoretical peak, due to interpreter overhead and lack of hardware optimizations.
- NumPy achieves significantly higher performance because it uses optimized BLAS libraries with vectorization and possible multi-threading.
- The performance scales consistently with 𝑂(𝑁3), as expected for DGEMM.
- The code benchmarks each implementation, computes mean runtime over multiple runs, and calculates GFLOPS using the total 2𝑁^3 floating-point operations.

### Task 2.5 Compare the performance results with the numpy matmul operation (that uses a BLAS library)

In [13]:
def dgemm_numpy_at(A, B, C):
    C += A @ B
    return C


def dgemm_numpy_matmul(A, B, C):
    C += np.matmul(A, B)
    return C
# =====================================
# BENCHMARK FUNCTION
# =====================================

def benchmark(func, A, B, C, N=None, repeats=5):
    times = []

    for _ in range(repeats):

        # Copy C for fresh computation
        if isinstance(C, np.ndarray):
            C_copy = C.copy()
        else:
            C_copy = [row[:] for row in C]

        start = time.perf_counter()

        if N is not None:
            func(A, B, C_copy, N)
        else:
            func(A, B, C_copy)

        end = time.perf_counter()
        times.append(end - start)

    return {
        "mean": mean(times),
        "std": stdev(times)
    }


# =====================================
# FLOPS CALCULATION
# =====================================

def compute_gflops(N, time_seconds):
    total_flops = 2 * (N ** 3)
    return (total_flops / time_seconds) / 1e9


def main():

    sizes = [100, 200, 400]
    repeats = 5

    for N in sizes:

        print("=" * 60)
        print(f"Matrix size: {N} x {N}")

        # Numpy matrices
        A_np = np.random.rand(N, N)
        B_np = np.random.rand(N, N)
        C_np = np.zeros((N, N))

        # Lists
        A_list = A_np.tolist()
        B_list = B_np.tolist()
        C_list = [[0.0] * N for _ in range(N)]

        # Arrays
        A_arr = [array('d', row) for row in A_np]
        B_arr = [array('d', row) for row in B_np]
        C_arr = [array('d', [0.0] * N) for _ in range(N)]

        # Run benchmarks
        results = {
            "Lists": benchmark(dgemm_lists, A_list, B_list, C_list, N, repeats),
            "Arrays": benchmark(dgemm_arrays, A_arr, B_arr, C_arr, N, repeats),
            "NumPy @": benchmark(dgemm_numpy_at, A_np, B_np, C_np, None, repeats),
            "NumPy matmul": benchmark(dgemm_numpy_matmul, A_np, B_np, C_np, None, repeats),
        }

        for name, stats in results.items():
            gflops = compute_gflops(N, stats["mean"])

            print(f"\n{name}:")
            print(f"  Mean Time: {stats['mean']:.6f} s")
            print(f"  Std Dev  : {stats['std']:.6f} s")
            print(f"  GFLOPS   : {gflops:.3f}")


In [14]:
!python compare_benchmark_dgemm_numpy.py 

Matrix size: 100 x 100

Lists:
  Mean Time: 0.069379 s
  Std Dev  : 0.009161 s
  GFLOPS   : 0.029

Arrays:
  Mean Time: 0.131431 s
  Std Dev  : 0.001206 s
  GFLOPS   : 0.015

NumPy @:
  Mean Time: 0.000029 s
  Std Dev  : 0.000021 s
  GFLOPS   : 69.626

NumPy matmul:
  Mean Time: 0.000020 s
  Std Dev  : 0.000001 s
  GFLOPS   : 102.215
Matrix size: 200 x 200

Lists:
  Mean Time: 0.551880 s
  Std Dev  : 0.010808 s
  GFLOPS   : 0.029

Arrays:
  Mean Time: 1.014351 s
  Std Dev  : 0.008298 s
  GFLOPS   : 0.016

NumPy @:
  Mean Time: 0.000111 s
  Std Dev  : 0.000054 s
  GFLOPS   : 144.502

NumPy matmul:
  Mean Time: 0.000087 s
  Std Dev  : 0.000004 s
  GFLOPS   : 183.416
Matrix size: 400 x 400

Lists:
  Mean Time: 4.919403 s
  Std Dev  : 0.020154 s
  GFLOPS   : 0.026

Arrays:
  Mean Time: 8.338602 s
  Std Dev  : 0.097893 s
  GFLOPS   : 0.015

NumPy @:
  Mean Time: 0.000632 s
  Std Dev  : 0.000178 s
  GFLOPS   : 202.617

NumPy matmul:
  Mean Time: 0.000572 s
  Std Dev  : 0.000032 s
  GFLOPS   

- Implements matrix multiplication (DGEMM) using four approaches: Pure Python lists (triple nested loops), Python array module ,NumPy, NumPy using np.matmul. Benchmarks performance for different matrix sizes (100×100, 200×200, 400×400) by measuring execution time multiple times and computing the mean and standard deviation. Calculates GFLOPS (Giga Floating Point Operations per Second) using the formula 2𝑁^3 , which represents the total floating-point operations in matrix multiplication.
- NumPy is dramatically faster than Python lists and arrays achieving over 200 GFLOPS, while pure Python stays around 0.02–0.03 GFLOPS. This shows the huge benefit of optimized linear algebra libraries.
- Performance gap increases with matrix size. As N grows, NumPy scales extremely well, while Python loop-based implementations become exponentially slower.
- @ and np.matmul perform almost identically, confirming they internally use the same highly optimized BLAS backend.
- Python arrays are not faster than lists in this case, because both still rely on slow Python-level loops rather than low-level optimized C/Fortran routines.